# Replication notebook
This notebook walks through the package step by step: data loading, feature construction, sample splitting, model estimation, and evaluation.

In [1]:
from __future__ import annotations
from pathlib import Path

import logging
from pandas.api.types import is_numeric_dtype
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from pandas import Timestamp
from typing import Any

from data_inputs import load_datashare, load_crsp_monthly, load_macro_monthly
from io_utils import save_parquet, ensure_dir, save_parquet, set_global_seed, setup_project_logger
from transformers import GroupedMedianImputer, GroupedQuantileTransformer, GroupedLagShiftTransformer
from dataset_builder import compute_missingness_for_characteristics, save_missingness_comparison_plot, save_missingness_three_comparison_plot, summary_stats_extended

from pathlib import Path


from config import (
    CacheConfig,
    TimeframeConfig,
    DataFilesConfig,
    RunControlConfig,
    SplitConfig,
    DataRegimeConfig,
    HyperGridConfig,
    ReproducibilityConfig,
    CharacteristicsFrequency,
    LoggingConfig,
    ExpandingWindowConfig,
    ModelSelectionConfig,
)

In [2]:
repro_cfg = ReproducibilityConfig(random_state=42)
cache_cfg = CacheConfig()
tf_cfg = TimeframeConfig()
data_cfg = DataFilesConfig()
run_ctrl_cfg = RunControlConfig()
split_cfg = SplitConfig()
regime_cfg = DataRegimeConfig(mode="full")
grid_cfg = HyperGridConfig(use_extended_grids=False)
freq_cfg = CharacteristicsFrequency()
log_cfg = LoggingConfig()
expand_cfg = ExpandingWindowConfig()
model_cfg = ModelSelectionConfig()


In [3]:

# Set global seed ONCE at the start - ensures reproducibility across all runs
set_global_seed(repro_cfg.random_state, repro_cfg.torch_deterministic)

logger, log_path = setup_project_logger(
    logger_name=log_cfg.logger_name,
    log_dir=log_cfg.log_dir,
    regime_mode=regime_cfg.mode,
    overwrite_log=log_cfg.overwrite_log,
    file_level_full=log_cfg.file_level_full,
    file_level_coding=log_cfg.file_level_coding,
    console_level_full=log_cfg.console_level_full,
    console_level_coding=log_cfg.console_level_coding,
)

logger.info("Starting experiment run.")
logger.info(f"Random seed: {repro_cfg.random_state}")
random_state=repro_cfg.random_state
logger.info(f"Cache dir: {cache_cfg.cache_dir}")
logger.info(f"Data regime: {regime_cfg.mode}")


2026-09-22 14:01:27 | INFO | eap_ml | Logger initialized.
2026-09-22 14:01:27 | INFO | eap_ml | Regime mode: full
2026-09-22 14:01:27 | INFO | eap_ml | Log file: logs\eap_ml_full_20260922_140127.log
2026-09-22 14:01:27 | INFO | eap_ml | Starting experiment run.
2026-09-22 14:01:27 | INFO | eap_ml | Random seed: 42
2026-09-22 14:01:27 | INFO | eap_ml | Cache dir: cache
2026-09-22 14:01:27 | INFO | eap_ml | Data regime: full


In [4]:
ensure_dir("output")
ensure_dir(cache_cfg.cache_dir)

In [5]:
complete_dataset_path=f"{cache_cfg.cache_dir}/complete_dataset.parquet"
descriptives_path=cache_cfg.descriptives_dir

feature_panel_path = f"{cache_cfg.cache_dir}/feature_panel_{regime_cfg.mode}.parquet"
feature_cols_path = f"{cache_cfg.cache_dir}/feature_cols_{regime_cfg.mode}.pkl"

In [6]:
datashare_path=data_cfg.datashare_path
crsp_path=data_cfg.crsp_monthly_path
macro_path=data_cfg.macro_path
out_path=complete_dataset_path
descriptives_path=descriptives_path
cache_enabled=cache_cfg.enabled
possible_crsp_cols=data_cfg.possible_crsp_cols
possible_marco_cols=data_cfg.possible_marco_cols
cols_chara=data_cfg.chara_cols
cols_vars_monthly=freq_cfg.cols_vars_monthly
cols_vars_quarterly=freq_cfg.cols_vars_quarterly
cols_vars_annual=freq_cfg.cols_vars_annual

In [7]:
logger.info(f"Building complete dataset, output: {out_path}")


logger.debug("Loading source datasets")

crsp = load_crsp_monthly(crsp_path, possible_crsp_cols)
macro = load_macro_monthly(macro_path, possible_marco_cols)
ds = load_datashare(datashare_path)


# Columns 3-96 in datashare.csv = 94 characteristics.
characteristic_cols = cols_chara
logger.debug(f"Identified {len(characteristic_cols)} characteristic columns")

logger.debug("Merging datashare with CRSP")
merged = ds.merge(
    crsp[["permno", "date", "ret", "dlret"]],
    on=["permno", "date"],
    how="inner",
    # validate="one_to_one",
)


logger.debug("Merging with macro data")
merged = merged.merge(macro, on="date", how="left")

merged = merged.sort_values(["permno", "date"]).reset_index(drop=True)

# ------------------------------------------------------------------
# excess_ret_total
# ------------------------------------------------------------------
logger.debug("Creating excess_ret_total")

has_return_data = (merged["ret"].notna() | merged["dlret"].notna())

merged["excess_ret_total"] = (
    (1.0 + merged["ret"].fillna(0.0))
    * (1.0 + merged["dlret"].fillna(0.0))
    - 1.0 - merged["rf"]
).where(has_return_data)    


summary_stats_extended(
    merged,
    date_col="date",
    output_path=descriptives_path,
    output_name="summary_loaded",
)
# ------------------------------------------------------------------
# Missingness visualisation and imputation
# ------------------------------------------------------------------
cols_chara_and_excess_ret_total = characteristic_cols + ["excess_ret_total"]

# Visualisation before imputation
missing_before = compute_missingness_for_characteristics(merged, cols_chara_and_excess_ret_total)

before_csv = f"{descriptives_path}/charas_missingness_before.csv"
missing_before.to_csv(before_csv, index=False)


logger.debug(f"Saved missingness before imputation: {before_csv}")

# Imputation
logger.info("Imputing missing characteristics using monthly cross-sectional medians")

imputer = GroupedMedianImputer(
    group_cols=["date"],
    value_cols=cols_chara_and_excess_ret_total,
    fallback="constant",
    fill_value=0.0
)

merged = imputer.fit_transform(merged)
merged = pd.DataFrame(merged)


# Visualisation after imputation
missing_after = compute_missingness_for_characteristics(merged, cols_chara_and_excess_ret_total)

after_csv = f"{descriptives_path}/charas_missingness_after.csv"
missing_after.to_csv(after_csv, index=False)


logger.debug(f"Saved missingness after imputation: {after_csv}")

# Comparison plot
comparison_jpg = f"{descriptives_path}/charas_missingness_comparison.jpg"
save_missingness_comparison_plot(
    missing_before,
    missing_after,
    comparison_jpg,
    title="Missing Data Percentage: Before vs After Imputation",
)
logger.debug(f"Saved missingness comparison plot: {comparison_jpg}")

summary_stats_extended(
    merged,
    date_col="date",
    output_path=descriptives_path,
    output_name="summary_imputed",
)


# ------------------------------------------------------------------
# Temporal shifts 
# ------------------------------------------------------------------
merged = merged.sort_values(["permno", "date"]).reset_index(drop=True)
shift_1month = ["excess_ret_total", *cols_vars_monthly]


transformer_1month = GroupedLagShiftTransformer(
    lag=-1,
    group_col="permno",
    value_cols=shift_1month,
    create_new_cols=False,
    rename_mode="lagged",
    suffix=None,
    original_suffix="_original",
)
transformer_3months = GroupedLagShiftTransformer(
    lag=-3,
    group_col="permno",
    value_cols=cols_vars_quarterly,
    create_new_cols=False,
    rename_mode="lagged",
    suffix=None,
    original_suffix="_original",
)
transformer_6months = GroupedLagShiftTransformer(
    lag=-6,
    group_col="permno",
    value_cols=cols_vars_annual,
    create_new_cols=False,
    rename_mode="lagged",
    suffix=None,
    original_suffix="_original",
)

logger.info("Creating 1-month shift.")
merged = transformer_1month.fit_transform(merged)
logger.debug("1-month shift created.")

logger.info("Creating 3-month shift.")
merged = transformer_3months.fit_transform(merged)
logger.debug("3-month shift created.")

logger.info("Creating 6-month shift.")
merged = transformer_6months.fit_transform(merged)
logger.debug("6-month shift created.")

merged = pd.DataFrame(merged)
logger.debug("Df after shifts recreated.")

summary_stats_extended(
    merged,
    date_col="date",
    output_path=descriptives_path,
    output_name="summary_shifted",
)


# Drop NaN
merged = merged.dropna(subset=cols_chara_and_excess_ret_total)


# Visualisation of missingness after temporal cuts
missing_after_drop = compute_missingness_for_characteristics(merged, cols_chara_and_excess_ret_total)

after_drop_csv = f"{descriptives_path}/charas_missingness_after_drop.csv"
missing_after_drop.to_csv(after_drop_csv, index=False)

logger.debug(f"Saved missingness after temporal cut: {after_drop_csv}")

# Comparison plot
comparison_three_jpg = f"{descriptives_path}/charas_missingness_comparison_three.jpg"
save_missingness_three_comparison_plot(
    missing_before,
    missing_after,
    missing_after_drop,
    comparison_three_jpg,
    title="Missing Data Percentage: Before vs After Imputation vs After Temporal Reduction",
)
logger.debug(f"Saved missingness comparison plot: {comparison_jpg}")

summary_stats_extended(
    merged,
    date_col="date",
    output_path=descriptives_path,
    output_name="summary_final_NaN_dropped"
)



2026-09-22 14:01:27 | INFO | eap_ml | Building complete dataset, output: cache/complete_dataset.parquet
2026-09-22 14:01:27 | DEBUG | eap_ml | Loading source datasets
2026-09-22 14:01:27 | DEBUG | eap_ml.data_inputs | Loading CRSP monthly data from C:/Coding/Project/data/crsp_monthly.csv
c:\Coding\thesis-project\data_inputs.py:121: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)
2026-09-22 14:01:35 | DEBUG | eap_ml.data_inputs | Standardizing date of CRSP
2026-09-22 14:01:46 | DEBUG | eap_ml.data_inputs | CRSP loaded: 4594389 rows
2026-09-22 14:01:46 | DEBUG | eap_ml.data_inputs | Loading macro data from C:/Coding/Project/data/Data2024_monthly_goyal.csv
2026-09-22 14:01:46 | DEBUG | eap_ml.data_inputs | Standardizing date of Macro
2026-09-22 14:01:46 | DEBUG | eap_ml.data_inputs | Macro loaded: 1848 rows, columns: ['date', 'tbl', 'rf', 'd/p', 'e/p', 'b/m', 'tms', 'dfy', 'ntis', 'svar']
2026-09-22 14:01:46 | DE

Summary statistics saved to: descriptives\summary_loaded.csv


2026-09-22 14:06:18 | DEBUG | eap_ml | Saved missingness before imputation: descriptives/charas_missingness_before.csv
2026-09-22 14:06:18 | INFO | eap_ml | Imputing missing characteristics using monthly cross-sectional medians
2026-09-22 14:07:26 | DEBUG | eap_ml | Saved missingness after imputation: descriptives/charas_missingness_after.csv
2026-09-22 14:07:28 | DEBUG | eap_ml | Saved missingness comparison plot: descriptives/charas_missingness_comparison.jpg


Summary statistics saved to: descriptives\summary_imputed.csv


2026-09-22 14:08:39 | INFO | eap_ml | Creating 1-month shift.
2026-09-22 14:08:42 | DEBUG | eap_ml | 1-month shift created.
2026-09-22 14:08:42 | INFO | eap_ml | Creating 3-month shift.
2026-09-22 14:08:44 | DEBUG | eap_ml | 3-month shift created.
2026-09-22 14:08:44 | INFO | eap_ml | Creating 6-month shift.
2026-09-22 14:08:48 | DEBUG | eap_ml | 6-month shift created.
2026-09-22 14:08:48 | DEBUG | eap_ml | Df after shifts recreated.


Summary statistics saved to: descriptives\summary_shifted.csv


2026-09-22 14:10:03 | DEBUG | eap_ml | Saved missingness after temporal cut: descriptives/charas_missingness_after_drop.csv
2026-09-22 14:10:03 | DEBUG | eap_ml | Saved missingness comparison plot: descriptives/charas_missingness_comparison.jpg
c:\Coding\thesis-project\dataset_builder.py:315: UserWarning: Column 'dlret' has 100% missing values
  warnings.warn(f"Column '{col}' has 100% missing values")


Summary statistics saved to: descriptives\summary_final_NaN_dropped.csv


,variable,dtype,min,max,range,nunique,sum,missing_count,missing_pct,first_obs_date,last_obs_date
0,permno,Int64,10000,93436,8.343600e+04,31726,2.187185e+11,0,0.0,1957-01-01,2021-06-01
1,date,datetime64[ns],1957-01-01 00:00:00,2021-06-01 00:00:00,NaN,774,NaN,0,0.0,1957-01-01,2021-06-01
2,mvel1,float64,0.0,2267638887.5,2.267639e+09,2480302,7.009809e+12,0,0.0,1957-01-01,2021-06-01
3,beta,float64,-0.79958,3.88742,4.687000e+00,3500349,3.960161e+06,0,0.0,1957-01-01,2021-06-01
4,betasq,float64,0.0,15.112035,1.511203e+01,3534099,5.499435e+06,0,0.0,1957-01-01,2021-06-01
...,...,...,...,...,...,...,...,...,...,...,...
104,tms,float64,-0.0365,0.0455,8.200000e-02,424,7.535328e+04,0,0.0,1957-01-01,2021-06-01
105,dfy,float64,0.0032,0.0338,3.060000e-02,165,4.017066e+04,0,0.0,1957-01-01,2021-06-01
106,ntis,float64,-0.055954,0.051161,1.071150e-01,768,3.125715e+04,0,0.0,1957-01-01,2021-06-01
107,svar,float64,0.000072,0.073153,7.308100e-02,682,9.472147e+03,0,0.0,1957-01-01,2021-06-01


In [8]:
# ------------------------------------------------------------------
# Quantile Transformation 
# ------------------------------------------------------------------

test_col = ["bm"]

scalar = GroupedQuantileTransformer(
    group_col="date",
    value_cols=test_col,
    output_range=(-1, 1),
    random_state=random_state,
)



logger.debug("Grouped Quantile Transformer test before")
transformed = scalar.fit_transform(merged)
logger.debug("Grouped Quantile Transformer test after")

transformed = pd.DataFrame(transformed)

summary_stats_extended(
    transformed,
    date_col="date",
    output_path=descriptives_path,
    output_name="summary_scaled_test",
)



2026-09-22 14:15:57 | DEBUG | eap_ml | Grouped Quantile Transformer test before
2026-09-22 14:16:53 | DEBUG | eap_ml | Grouped Quantile Transformer test after
c:\Coding\thesis-project\dataset_builder.py:315: UserWarning: Column 'dlret' has 100% missing values
  warnings.warn(f"Column '{col}' has 100% missing values")


Summary statistics saved to: descriptives\summary_scaled_test.csv


,variable,dtype,min,max,range,nunique,sum,missing_count,missing_pct,first_obs_date,last_obs_date
0,permno,Int64,10000,93436,8.343600e+04,31726,2.187185e+11,0,0.0,1957-01-01,2021-06-01
1,date,datetime64[ns],1957-01-01 00:00:00,2021-06-01 00:00:00,NaN,774,NaN,0,0.0,1957-01-01,2021-06-01
2,mvel1,float64,0.0,2267638887.5,2.267639e+09,2480302,7.009809e+12,0,0.0,1957-01-01,2021-06-01
3,beta,float64,-0.79958,3.88742,4.687000e+00,3500349,3.960161e+06,0,0.0,1957-01-01,2021-06-01
4,betasq,float64,0.0,15.112035,1.511203e+01,3534099,5.499435e+06,0,0.0,1957-01-01,2021-06-01
...,...,...,...,...,...,...,...,...,...,...,...
104,tms,float64,-0.0365,0.0455,8.200000e-02,424,7.535328e+04,0,0.0,1957-01-01,2021-06-01
105,dfy,float64,0.0032,0.0338,3.060000e-02,165,4.017066e+04,0,0.0,1957-01-01,2021-06-01
106,ntis,float64,-0.055954,0.051161,1.071150e-01,768,3.125715e+04,0,0.0,1957-01-01,2021-06-01
107,svar,float64,0.000072,0.073153,7.308100e-02,682,9.472147e+03,0,0.0,1957-01-01,2021-06-01


In [9]:
# ------------------------------------------------------------------
# Quantile Transformation 
# ------------------------------------------------------------------

scalar = GroupedQuantileTransformer(
    group_col="date",
    value_cols=cols_chara,
    output_range=(-1, 1),
    random_state=random_state,
)

transformed_final = scalar.fit_transform(merged)
transformed_final = pd.DataFrame(transformed_final)

summary_stats_extended(
    transformed_final,
    date_col="date",
    output_path=descriptives_path,
    output_name="summary_scaled",
)



c:\Coding\thesis-project\dataset_builder.py:315: UserWarning: Column 'dlret' has 100% missing values
  warnings.warn(f"Column '{col}' has 100% missing values")


Summary statistics saved to: descriptives\summary_scaled.csv


,variable,dtype,min,max,range,nunique,sum,missing_count,missing_pct,first_obs_date,last_obs_date
0,permno,Int64,10000,93436,83436.000000,31726,2.187185e+11,0,0.0,1957-01-01,2021-06-01
1,date,datetime64[ns],1957-01-01 00:00:00,2021-06-01 00:00:00,NaN,774,NaN,0,0.0,1957-01-01,2021-06-01
2,mvel1,float64,-1.0,1.0,2.000000,3889976,-1.884524e+01,0,0.0,1957-01-01,2021-06-01
3,beta,float64,-1.0,1.0,2.000000,3497656,8.896370e+00,0,0.0,1957-01-01,2021-06-01
4,betasq,float64,-1.0,1.0,2.000000,3531460,2.713981e+02,0,0.0,1957-01-01,2021-06-01
...,...,...,...,...,...,...,...,...,...,...,...
104,tms,float64,-0.0365,0.0455,0.082000,424,7.535328e+04,0,0.0,1957-01-01,2021-06-01
105,dfy,float64,0.0032,0.0338,0.030600,165,4.017066e+04,0,0.0,1957-01-01,2021-06-01
106,ntis,float64,-0.055954,0.051161,0.107115,768,3.125715e+04,0,0.0,1957-01-01,2021-06-01
107,svar,float64,0.000072,0.073153,0.073081,682,9.472147e+03,0,0.0,1957-01-01,2021-06-01


In [10]:
merged = transformed_final

In [11]:
logger.info(f"Complete dataset built: {merged.shape}")
save_parquet(merged, out_path, enabled=cache_enabled)

2026-09-22 15:55:50 | INFO | eap_ml | Complete dataset built: (3922932, 109)


In [ ]:
merged